## Mount Google Drive

Run the following cell to mount your Google Drive. This will allow the notebook to access files stored in your Drive.

## Unzip Training Data

Now, specify the path to your zip file in Google Drive. This path should start with `/content/drive/MyDrive/`. Once you've updated the `zip_file_path` variable, run the next cell to unzip the file.

In [2]:
# Replace 'path/to/your/training_data.zip' with the actual path to your zip file in Google Drive
zip_file_path = '/content/drive/MyDrive/Fashion144k_v1.zip'

# Specify the directory where you want to extract the contents
extraction_path = './training_data'

import os
import zipfile

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Successfully unzipped '{zip_file_path}' to '{extraction_path}'")

# List the contents of the extracted directory (optional, for verification)
print("Contents of the extracted directory:")
print(os.listdir(extraction_path))

In [ ]:
!git clone https://github.com/bugsNburgers/AI-based-personal-stylist.git /content/AI-based-personal-stylist


In [ ]:
import os
print(os.listdir('/content/AI-based-personal-stylist'))
print(os.listdir('/content/AI-based-personal-stylist/real_image_pipeline'))

In [ ]:
print(os.listdir('./training_data/Fashion144k_v1'))

In [ ]:
import os
base = './training_data/Fashion144k_v1'

print("feat/:", os.listdir(os.path.join(base, 'feat')))
print()
print("photos/ (first 5):", os.listdir(os.path.join(base, 'photos'))[:5])
print("photos/ count:", len(os.listdir(os.path.join(base, 'photos'))))
print()
with open(os.path.join(base, 'photos.txt')) as f:
    lines = f.readlines()
print("photos.txt line count:", len(lines))
print("photos.txt first 3 lines:", lines[:3])

In [ ]:
REPO_DIR   = '/content/AI-based-personal-stylist'
DATA_ROOT  = './training_data/Fashion144k_v1'
PHOTOS_LIST = f'{DATA_ROOT}/photos.txt'
IMAGES_DIR  = f'{DATA_ROOT}/photos'
SPLIT_MAT     = f'{DATA_ROOT}/split.mat'
RELVOTES_MAT  = f'{DATA_ROOT}/feat/relvotes.mat'

# Outputs should go to Drive so they persist across Colab sessions
OUTPUT_ROOT = '/content/drive/MyDrive/Fashion144k_outputs'
CHECKPOINT_DIR = '/content/drive/MyDrive/Fashion144k_gnn_checkpoints'

In [ ]:
!git clone https://github.com/bugsNburgers/AI-based-personal-stylist.git {REPO_DIR}
!pip install -q -r {REPO_DIR}/real_image_pipeline/requirements.txt


In [ ]:
!python 01_segment_batch.py \
    --repo_dir {REPO_DIR} \
    --photos_list {PHOTOS_LIST} \
    --images_dir {IMAGES_DIR} \
    --output_root {OUTPUT_ROOT} \
    --checkpoint {OUTPUT_ROOT}/segment_checkpoint.txt \
    --limit 2000

In [ ]:
!python 02_outfit_dataset.py \
    --output_root {OUTPUT_ROOT} \
    --split_mat {SPLIT_MAT} \
    --relvotes_mat {RELVOTES_MAT}

In [ ]:
import os, json
from importlib.util import spec_from_file_location, module_from_spec

spec = spec_from_file_location("outfit_dataset", "02_outfit_dataset.py")
outfit_dataset = module_from_spec(spec)
spec.loader.exec_module(outfit_dataset)

relvotes = outfit_dataset.load_relvotes(RELVOTES_MAT)

# use indices that actually exist in the pilot output
processed_indices = sorted(int(d) for d in os.listdir(OUTPUT_ROOT) if d.isdigit())
print(f"{len(processed_indices)} outfits actually segmented so far")

ds = outfit_dataset.OutfitDataset(processed_indices[:50], OUTPUT_ROOT, relvotes)
print(f"{len(ds)} valid (>=2 garments) out of first 50 processed indices")

if len(ds) > 0:
    item = ds[0]
    print(f"outfit_idx={item['outfit_idx']} x.shape={item['x'].shape} "
          f"categories={item['categories']} fashion_score={item['fashion_score'].item():.2f}")

In [ ]:
!python 01_segment_batch.py \
    --repo_dir {REPO_DIR} \
    --photos_list {PHOTOS_LIST} \
    --images_dir {IMAGES_DIR} \
    --output_root {OUTPUT_ROOT} \
    --checkpoint {OUTPUT_ROOT}/segment_checkpoint.txt \
    --limit 200000

In [ ]:
!python 03_train_gnn.py \
    --output_root {OUTPUT_ROOT} \
    --split_mat {SPLIT_MAT} \
    --relvotes_mat {RELVOTES_MAT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5 --grad_accum 16